# Citify Phase F v4 — 全自治体 Reinfolib データ拡張 (Google Colab 実行用)

Phase F v1/v3 で 45 自治体に限定していた Reinfolib データ (取引価格 / 避難所 / 将来人口 / 医療 / 保育) を、**全 1,794 自治体** (or 任意のサブセット) に拡張する。

## なぜ Colab で実行するか
- Reinfolib API は rate_limit 1 req/sec を強制 → 全自治体だと数十時間バッチ放置
- ローカル PC を占有せず Colab セッション (最長 12h) で実行
- Google Drive にチェックポイント保存で resume 可能

## 実行ステップ
1. パッケージ install + Drive mount + API キー設定
2. Citify リポジトリ clone (scrapers/reinfolib をそのまま import)
3. 国土数値情報 N03 をダウンロード → 自治体 centroid を計算 → `municipality_centroids.csv` 生成
4. master.csv + centroids を merge → targets DataFrame
5. **バッチ実行** (checkpoint 付き、途中で停止しても resume 可)
6. 完成 CSV を Drive 経由でローカルに DL → `bq load` で BQ MERGE

## スコープ判断
1 自治体あたり API レスポンス約 100 req × 1 秒 ≒ 100 秒。1794 自治体だと約 50 時間。Colab 1 セッション 12h で進められる量は ~400 自治体。**スコープ縮小推奨**:

| Scope | 自治体数 | 推定時間 |
|---|---|---|
| 政令市 + 中核市 + 特別区 + 都道府県 (Phase F v3 拡張版) | ~200 | 5-6h |
| + 主要市町村 (人口 10 万以上) | ~300 | 8-10h |
| 全 1794 自治体 (Pro+ で 24h セッション × 3 必要) | 1794 | 50h |

デモ的には ~200 自治体で十分。Notebook の `TARGET_SCOPE` 変数で切り替え可能。

## Cell 1: パッケージインストール

In [ ]:
!pip install -q httpx pandas tqdm geopandas pyogrio

## Cell 2: Google Drive mount + API キー設定

Colab 左サイドバーの 🔑 (Secret) で `REINFOLIB_API_KEY` を追加してから実行。

In [ ]:
import os
from pathlib import Path

# Drive mount (チェックポイント / 出力 CSV の保存先)
from google.colab import drive  # type: ignore
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/citify')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'work dir: {WORK_DIR}')

# API キー (Colab Secret から)
from google.colab import userdata  # type: ignore
os.environ['REINFOLIB_API_KEY'] = userdata.get('REINFOLIB_API_KEY').strip()
print('REINFOLIB_API_KEY:', 'OK' if os.environ.get('REINFOLIB_API_KEY') else 'NOT SET')

## Cell 3: Citify リポジトリ clone (scrapers/reinfolib を再利用)

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('/content/citify')
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/yujmatsu/citify.git /content/citify
else:
    !cd /content/citify && git pull --rebase --autostash

sys.path.insert(0, str(REPO_DIR))

from scrapers.reinfolib.client import ReinfolibClient, ReinfolibAPIError
from scrapers.reinfolib.parsers.xit001 import aggregate_used_apartments
from scrapers.reinfolib.parsers.xgt001 import aggregate_shelters
from scrapers.reinfolib.parsers.xkt013 import aggregate_future_population
from scrapers.reinfolib.parsers.xkt010 import aggregate_medical
from scrapers.reinfolib.parsers.xkt007 import aggregate_childcare

print('imports OK')

## Cell 4: 自治体 centroid 計算 (N03 行政区域 → lat/lng)

国土数値情報 N03 (全国行政区域ポリゴン Shapefile、約 400MB) をダウンロードして自治体の重心座標を計算。1 度実行すれば `municipality_centroids.csv` (~70KB) を Drive に保存して以後 reuse 可能。

In [ ]:
CENTROIDS_CSV = WORK_DIR / 'municipality_centroids.csv'

if CENTROIDS_CSV.exists():
    print(f'既存ファイル使用: {CENTROIDS_CSV}')
else:
    print('N03 から centroid 計算を開始...')
    import zipfile
    import urllib.request

    N03_URL = 'https://nlftp.mlit.go.jp/ksj/gml/data/N03/N03-2024/N03-20240101_GML.zip'
    N03_ZIP = Path('/content/n03.zip')
    N03_DIR = Path('/content/n03')

    if not N03_ZIP.exists():
        print(f'downloading {N03_URL} ...')
        urllib.request.urlretrieve(N03_URL, N03_ZIP)
        print(f'downloaded: {N03_ZIP.stat().st_size / 1024 / 1024:.1f} MB')

    if not N03_DIR.exists():
        with zipfile.ZipFile(N03_ZIP) as z:
            z.extractall(N03_DIR)
    shp_files = list(N03_DIR.rglob('*.shp'))
    print(f'shapefiles: {[str(p) for p in shp_files]}')

    import geopandas as gpd
    gdf = gpd.read_file(shp_files[0], encoding='shift_jis')
    print(f'features: {len(gdf)}, columns: {gdf.columns.tolist()}')

    # N03_007 = 5 桁市区町村コード
    # WGS84 (EPSG:4326) で centroid → lat/lng
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    else:
        gdf = gdf.to_crs('EPSG:4326')
    # 投影系で centroid を計算する方が幾何的に正確だが、ハッカソンスコープでは WGS84 の簡易 centroid で十分
    gdf['centroid'] = gdf.geometry.centroid
    gdf['lat'] = gdf['centroid'].y
    gdf['lng'] = gdf['centroid'].x

    # 同じ市区町村が複数 polygon (飛地等) の場合は最大面積のものを採用
    gdf['area_m2'] = gdf.geometry.to_crs('EPSG:6933').area  # 等積投影で面積計算
    gdf = gdf.sort_values('area_m2', ascending=False).drop_duplicates(subset='N03_007', keep='first')

    out = gdf[['N03_007', 'N03_001', 'N03_003', 'N03_004', 'lat', 'lng']].rename(
        columns={
            'N03_007': 'municipality_code',
            'N03_001': 'prefecture',
            'N03_003': 'subprefecture',
            'N03_004': 'name',
        }
    )
    out.to_csv(CENTROIDS_CSV, index=False)
    print(f'saved {len(out)} centroids to {CENTROIDS_CSV}')

import pandas as pd
df_centroids = pd.read_csv(CENTROIDS_CSV, dtype={'municipality_code': str})
print(f'centroids: {len(df_centroids)} 件')
df_centroids.head()

## Cell 5: targets DataFrame 構築 (master.csv + centroids)

各自治体について `(municipality_code, name, xit001_method, xit001_param, lat, lng)` を決定。
- `XX000` (都道府県): `area=XX`
- `XX100/XX130/XX140/XX150` (政令市親): `city_sum=XXNNN-XXMMM` (子区範囲)
- その他: `city=XXXXX`

`TARGET_SCOPE` で対象を絞り込む:
- `'small'`: 政令市 + 中核市 + 特別区 + 都道府県 ≒ 200 (5-6h)
- `'medium'`: + 中核市候補・特例市 ≒ 300 (8-10h)
- `'all'`: 全 1794 自治体 (50h、分割実行必要)

In [ ]:
import pandas as pd

TARGET_SCOPE = 'small'  # 'small' / 'medium' / 'all'

df_master = pd.read_csv(
    REPO_DIR / 'infra/seed/municipality_master.csv',
    dtype={'municipality_code': str},
)
df_master['municipality_code'] = df_master['municipality_code'].str.zfill(5)

# code → 子区 list (政令市親用)
def find_children(code: str) -> list[str]:
    prefix3 = code[:3]
    children = df_master[
        (df_master['municipality_code'].str.startswith(prefix3))
        & (df_master['municipality_code'] != code)
    ]
    return sorted(children['municipality_code'].tolist())


def classify_method(code: str) -> tuple[str, str]:
    suffix = code[2:]
    if suffix == '000':
        return ('area', code[:2])
    if suffix in ('100', '130', '140', '150'):
        children = find_children(code)
        if children:
            return ('city_sum', f'{children[0]}-{children[-1]}')
        return ('city', code)
    return ('city', code)


df_centroids_idx = df_centroids.set_index('municipality_code')[['lat', 'lng']]

targets = []
for _, row in df_master.iterrows():
    code = row['municipality_code']
    if code == '00000':
        continue
    name = row.get('name', '')
    pref = row.get('prefecture', '')
    if code not in df_centroids_idx.index:
        continue
    lat = df_centroids_idx.loc[code, 'lat']
    lng = df_centroids_idx.loc[code, 'lng']
    method, param = classify_method(code)
    targets.append({
        'municipality_code': code,
        'name': name,
        'prefecture': pref,
        'xit001_method': method,
        'xit001_param': param,
        'center_lat': str(lat),
        'center_lng': str(lng),
    })

df_targets_all = pd.DataFrame(targets)

# Scope フィルタ
def is_in_scope(code: str, scope: str) -> bool:
    suffix = code[2:]
    if scope == 'all':
        return True
    # small / medium とも以下は include
    if suffix == '000':
        return True  # 都道府県
    if suffix in ('100', '130', '140', '150'):
        return True  # 政令市親
    if code.startswith('13') and 100 < int(suffix) < 124:
        return True  # 東京 23 区 (13101-13123)
    # 中核市候補: コードの一の位が 1 で suffix が 201-399 (簡易判定)
    if scope in ('small', 'medium'):
        if int(suffix) >= 201 and int(suffix) <= 250:
            return True
    return False


df_targets = df_targets_all[df_targets_all['municipality_code'].apply(lambda c: is_in_scope(c, TARGET_SCOPE))].reset_index(drop=True)
print(f'TARGET_SCOPE={TARGET_SCOPE}: {len(df_targets)} 自治体')
df_targets.head(10)

## Cell 6: バッチ実行 (resume 対応)

各自治体について 5 API (XIT001 + XGT001 + XKT013 + XKT010 + XKT007) を取得し、1 行ずつ CSV に append。途中停止しても次回起動時に続きから再開できる。

In [ ]:
import datetime as dt
import csv
import logging
import time
from tqdm.auto import tqdm

logging.basicConfig(level=logging.WARNING)  # tqdm progress を見やすくするため INFO を抑制

OUTPUT_CSV = WORK_DIR / f'reinfolib_normalized_full_{TARGET_SCOPE}.csv'
OUTPUT_COLUMNS = [
    'municipality_code',
    'used_apartment_median_price_man_yen', 'used_apartment_sample_size',
    'used_apartment_median_unit_price_yen', 'used_apartment_avg_building_age',
    'emergency_shelter_count', 'emergency_shelter_official_link',
    'population_2025_estimated', 'population_2050_estimated', 'population_change_2025_2050_pct',
    'medical_facility_count', 'medical_hospital_count', 'medical_clinic_count',
    'childcare_facility_count', 'kindergarten_count', 'nursery_count',
    'reinfolib_loaded_at', 'reinfolib_source_url',
]
SOURCE_URL = 'https://www.reinfolib.mlit.go.jp/'

# 既存 CSV から完了コードを load (resume 対応)
completed_codes: set[str] = set()
if OUTPUT_CSV.exists():
    with OUTPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for r in reader:
            completed_codes.add(r['municipality_code'])
    print(f'resume: {len(completed_codes)} 自治体は完了済 (skip)')

todo = df_targets[~df_targets['municipality_code'].isin(completed_codes)]
print(f'残: {len(todo)} 自治体 (合計 {len(df_targets)})')

In [ ]:
def fetch_one_municipality(client: ReinfolibClient, target: dict) -> dict:
    code = target['municipality_code']
    method = target['xit001_method']
    param = target['xit001_param']
    lat = float(target['center_lat'])
    lng = float(target['center_lng'])

    out = {'municipality_code': code}

    # XIT001 取引価格 (4Q)
    try:
        trades = client.fetch_trades_4quarters(method, param)
        out.update(aggregate_used_apartments(trades))
    except Exception as e:
        print(f'  xit001 fail code={code}: {e}')
        out.update({
            'used_apartment_median_price_man_yen': None, 'used_apartment_sample_size': 0,
            'used_apartment_median_unit_price_yen': None, 'used_apartment_avg_building_age': None,
        })

    # XGT001 避難所 (z=11, 9 tiles)
    try:
        f1 = client.fetch_geojson_tiles('XGT001', lng, lat, z=11, radius=1)
        out.update(aggregate_shelters(f1, lat, lng))
    except Exception as e:
        print(f'  xgt001 fail code={code}: {e}')
        out.update({
            'emergency_shelter_count': 0,
            'emergency_shelter_official_link': f'https://disaportal.gsi.go.jp/hazardmap/maps/index.html?ll={lat},{lng}&z=12',
        })

    # XKT013 将来人口 (z=11, 9 tiles)
    try:
        f2 = client.fetch_geojson_tiles('XKT013', lng, lat, z=11, radius=1)
        out.update(aggregate_future_population(f2))
    except Exception as e:
        print(f'  xkt013 fail code={code}: {e}')
        out.update({
            'population_2025_estimated': None, 'population_2050_estimated': None,
            'population_change_2025_2050_pct': None,
        })

    # XKT010 医療機関 (z=13, 25 tiles)
    try:
        f3 = client.fetch_geojson_tiles('XKT010', lng, lat, z=13, radius=2)
        out.update(aggregate_medical(f3))
    except Exception as e:
        print(f'  xkt010 fail code={code}: {e}')
        out.update({
            'medical_facility_count': None, 'medical_hospital_count': None, 'medical_clinic_count': None,
        })

    # XKT007 保育園・幼稚園 (z=13, 25 tiles、city_sum は admin filter なし)
    try:
        f4 = client.fetch_geojson_tiles('XKT007', lng, lat, z=13, radius=2)
        if method == 'city_sum':
            out.update(aggregate_childcare(f4, municipality_code=None))
        else:
            out.update(aggregate_childcare(f4, municipality_code=code))
    except Exception as e:
        print(f'  xkt007 fail code={code}: {e}')
        out.update({
            'childcare_facility_count': None, 'kindergarten_count': None, 'nursery_count': None,
        })

    out['reinfolib_loaded_at'] = dt.datetime.now(dt.UTC).isoformat()
    out['reinfolib_source_url'] = SOURCE_URL
    return out


# CSV を append モードで開く (resume 対応、ヘッダなしならまず header 書込)
write_header = not OUTPUT_CSV.exists()
with ReinfolibClient(rate_limit_sec=1.0) as client, OUTPUT_CSV.open('a', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
    if write_header:
        writer.writeheader()
    for _, target in tqdm(todo.iterrows(), total=len(todo), desc='reinfolib fetch'):
        try:
            row = fetch_one_municipality(client, target.to_dict())
            writer.writerow(row)
            f.flush()
        except Exception as e:
            print(f'FAIL code={target["municipality_code"]} err={e}')

print(f'done. output: {OUTPUT_CSV}')
print(f'rows: {sum(1 for _ in OUTPUT_CSV.open("r")) - 1}')

## Cell 7: ローカルで実行する次のステップ

Drive に保存した CSV をローカルに DL してから、Citify リポジトリで:

```bash
# 1. Drive から CSV を DL (Google Drive App or browser から手動 DL)
# 2. リポジトリの seed/ に置く
cp ~/Downloads/reinfolib_normalized_full_small.csv /home/yujmatsu/projects/citify/infra/seed/

# 3. BQ MERGE (既存 load_reinfolib_stats.py を流用)
cd /home/yujmatsu/projects/citify/apps/api
.venv/bin/python -m scripts.load_reinfolib_stats \
  --input ../../infra/seed/reinfolib_normalized_full_small.csv \
  --dry-run

# 4. 本投入
.venv/bin/python -m scripts.load_reinfolib_stats \
  --input ../../infra/seed/reinfolib_normalized_full_small.csv

# 5. 確認
bq query --use_legacy_sql=false "
SELECT municipality_code, municipality_name,
       used_apartment_median_price_man_yen, population_change_2025_2050_pct,
       medical_facility_count, childcare_facility_count
FROM \`citify-dev.citify_curated.municipality_stats\`
WHERE used_apartment_median_price_man_yen IS NOT NULL
ORDER BY population_change_2025_2050_pct ASC
LIMIT 20
"

# 6. commit + push (CSV 含めても良いし、含めず BQ だけ更新でも良い)
git add infra/seed/reinfolib_normalized_full_small.csv
git commit -m 'feat(plan-a-f-v4): Reinfolib 全自治体拡張 (Colab バッチ)'
git push origin main
```

Cloud Run 再 deploy 不要 (BQ 列はすでに Phase F v3 で追加済、データだけ更新)。

## チェックポイント / 途中失敗時の対応
- `reinfolib_normalized_full_{scope}.csv` は append モードで書かれている → ランタイム切断 / 12h timeout 時も保存済
- 再起動後、Cell 6 を再実行すれば完了済コードは自動 skip
- 個別 API の失敗 (`fail code=XXXX`) はその列だけ None で記録、次の自治体に進む

## トラブルシューティング
- `quota_exceeded`: Reinfolib 側で 1 日のリクエスト上限に達した可能性 → 24 時間後に resume
- N03 ダウンロードが遅い: Cell 4 で 5-10 分かかる、放置で OK
- centroid が海上に出る (沿岸自治体): N03 ポリゴンが複雑な形状の場合 centroid が陸地外になる場合がある → 取得した API レスポンスが空になる可能性。データ豊富な API (XGT001/XKT013) はタイル数で吸収可能